[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day5_live.ipynb)

# Day 5 · 강의 — 이미지 분류

합성곱으로 만들고 남이 배운 것을 가져온다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 필터를 직접 통과시켜 본다

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)

# 자료를 가져온다 — 드라이브 폴더를 먼저 보고, 없거나 막히면 원래 자리에서 받는다
DRIVE = '1GqiActlNF0uv9D_1GPCRGMsWFcr_EQUH'   # 강사가 올려 둔 CIFAR-10
def get_cifar():
    import os
    if not os.path.isdir('data/cifar-10-batches-py'):
        os.system('pip install -q gdown')
        os.system(f'gdown --folder {DRIVE} -O data/cifar-10-batches-py -q')
    try:                                       # 드라이브에서 받아졌으면 이 줄이 통과한다
        return (datasets.CIFAR10('data', train=True,  download=False, transform=transforms.ToTensor()),
                datasets.CIFAR10('data', train=False, download=False, transform=transforms.ToTensor()))
    except Exception as e:                     # 안 되면 원래 자리에서 받는다 (1~2분)
        print('드라이브를 못 써서 원래 자리에서 받는다 —', type(e).__name__)
        return (datasets.CIFAR10('data', train=True,  download=True, transform=transforms.ToTensor()),
                datasets.CIFAR10('data', train=False, download=True, transform=transforms.ToTensor()))

train, test = get_cifar()

# 5,000장만 떼어 쓴다. Subset 은 데이터에서 일부만 골라 주는 것이다.
# 전체 5만 장으로 돌리면 한 번에 몇 분씩 걸려 여러 번 비교하기 어렵다.
small      = Subset(train, range(5000))
small_test = Subset(test,  range(1000))
loader      = DataLoader(small,      batch_size=128, shuffle=True)
test_loader = DataLoader(small_test, batch_size=500)

names = train.classes
print(names)

In [ ]:
def fit(model, ld=None, epochs=6, lr=0.001):
    """학습 루프 다섯 줄을 함수로 묶어 둔 것 — 1주차에 만든 것과 같다"""
    ld = ld if ld is not None else loader
    torch.manual_seed(42)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in ld:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model

def score(model, ld=None):
    """학습에 안 쓴 자료로 재는 정확도"""
    ld = ld if ld is not None else test_loader
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in ld:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total

def count(model):
    """배울 계수가 몇 개인지"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
img, label = test[13]
print('모양   ', img.shape)
print('이름   ', names[label])
print('값 범위', float(img.min()), '~', float(img.max()))

plt.imshow(img.permute(1, 2, 0))   # imshow 는 [행, 열, 색] 순서를 원한다
plt.axis('off'); plt.show()

# 색 세 면의 평균으로 흑백 한 장을 만들어 둔다. 필터 실습에서 이것을 쓴다.
gray = img.mean(0)
print('흑백 모양', gray.shape)

In [ ]:
def apply_filter(k, image=None):
    """3x3 필터 하나를 흑백 사진에 통과시켜 결과를 돌려준다"""
    image = image if image is not None else gray
    c = nn.Conv2d(1, 1, 3, padding=1, bias=False)
    c.weight.data[0, 0] = torch.tensor(k, dtype=torch.float)
    with torch.no_grad():
        return c(image.view(1, 1, 32, 32))[0, 0]

vert = apply_filter([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
print('세로 윤곽선 범위 %+.0f ~ %+.0f' % (vert.min() * 255, vert.max() * 255))

> **실습문제 1.** `img` 에서 **빨강 면 하나만** 꺼내 `red` 에 담는다. 모양이 `[3, 32, 32]` 이고 0번 축이 색이므로 그 축의 0번을 고른다.

In [ ]:
red = img[0]

print(red.shape)
assert tuple(red.shape) == (32, 32), f'32x32 여야 한다: {tuple(red.shape)}'
plt.imshow(red, cmap='Reds'); plt.axis('off'); plt.show()

> **실습문제 2.** 컬러 사진(채널 3)을 받아 **특징 지도 32장**을 내놓는 합성곱 층을 만든다. 창은 3×3, `padding=1` 로 가로세로를 유지한다.

In [ ]:
conv = nn.Conv2d(3, 32, 3, padding=1)

out = conv(img.unsqueeze(0))   # unsqueeze(0) 은 사진 한 장을 배치 하나로 감싸는 것
print(out.shape)
assert tuple(out.shape) == (1, 32, 32, 32), f'[1,32,32,32] 여야 한다: {tuple(out.shape)}'

> **실습문제 3.** `padding` 을 **0** 으로 바꾸면 가로세로가 어떻게 되는지 확인한다. 창이 사진 안에만 들어가야 하므로 양쪽 한 줄씩 못 쓴다.

In [ ]:
c0 = nn.Conv2d(3, 8, 3, padding=0)
print(c0(img.unsqueeze(0)).shape)

s = tuple(c0(img.unsqueeze(0)).shape)
assert s == (1, 8, 30, 30), f'[1,8,30,30] 이어야 한다: {s}'
print('32 였던 것이', s[2], '가 됐다')

## 2. shape 를 따라간다

> **실습문제 4.** `Conv2d` → `ReLU` → `MaxPool2d` 한 묶음을 통과시킨 뒤 모양을 찍는다. 채널은 3에서 16으로, 가로세로는 32에서 절반이 된다.

In [ ]:
block = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2))
print(block(img.unsqueeze(0)).shape)

s = tuple(block(img.unsqueeze(0)).shape)
assert s == (1, 16, 16, 16), f'[1,16,16,16] 이어야 한다: {s}'

> **실습문제 5.** `MaxPool2d` 를 빼고 `Flatten` 뒤에 `Linear(64*8*8, 10)` 을 붙이면 에러가 난다. 일부러 내 보고 **메시지의 네 숫자**를 읽는다.

In [ ]:
bad = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
try:
    bad(img.unsqueeze(0))
except RuntimeError as e:
    msg = str(e)
    print(msg)

assert 'msg' in dir() and 'shapes cannot be multiplied' in msg, '에러가 나야 정상이다'
print('들어온 칸 수 65536 · 적어 둔 칸 수 4096 — 풀링을 빼서 절반으로 줄지 않았다')

## 3. CNN 을 만들어 돌린다

In [ ]:
torch.manual_seed(42)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3072, 128), nn.ReLU(), nn.Linear(128, 10))
fit(mlp)
print('펴서 Linear  정확도 %.4f  계수 %d' % (score(mlp), count(mlp)))

# 비교 기준이 될 CNN 도 미리 한 벌 만들어 둔다. 아래 문제들이 이것과 견준다.
torch.manual_seed(42)
cnn = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
fit(cnn)
print('CNN          정확도 %.4f  계수 %d' % (score(cnn), count(cnn)))

> **실습문제 6.** 준비 셀의 CNN 을 **직접 다시 써서** `my_cnn` 을 만든다. 채널은 3 → 32 → 64, 가로세로는 32 → 16 → 8 이므로 마지막 `Linear` 의 입력은 `64*8*8` 이다. 학습까지 시켜 준비 셀과 같은 값이 나오는지 본다.

In [ ]:
torch.manual_seed(42)
my_cnn = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
fit(my_cnn)
print('정확도 %.4f  계수 %d' % (score(my_cnn), count(my_cnn)))

assert count(my_cnn) == 60362, f'계수가 60,362 여야 한다: {count(my_cnn)}'
assert score(my_cnn) == score(cnn), '준비 셀과 같은 값이 나와야 한다'
print('계수는 %d 분의 1 인데 정확도는 올랐다' % (count(mlp) // count(my_cnn)))

## 4. 이미 배운 모델을 가져온다

In [ ]:
from torchvision import models

net = models.resnet18(weights='DEFAULT')
print('전체 계수', sum(p.numel() for p in net.parameters()))
print('마지막 층 ', net.fc)

In [ ]:
T224 = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

big_train = datasets.CIFAR10('data', train=True,  transform=T224)
big_test  = datasets.CIFAR10('data', train=False, transform=T224)

backbone = models.resnet18(weights='DEFAULT')
backbone.fc = nn.Identity()      # 마지막 층을 없애는 대신 그대로 내놓게 한다
backbone.eval()

def features(ds, n):
    """사진 n 장을 얼린 앞쪽에 한 번 통과시켜 512칸씩 뽑아 둔다"""
    X, Y = [], []
    with torch.no_grad():
        for xb, yb in DataLoader(Subset(ds, range(n)), batch_size=64):
            X.append(backbone(xb)); Y.append(yb)
    return torch.cat(X), torch.cat(Y)

# 6,000장을 224x224 로 한 번 통과시킨다. GPU 면 30초, CPU 면 몇 분 걸린다.
# 런타임 유형을 GPU 로 바꿔 두면 훨씬 빠르다.
Xtr, Ytr = features(big_train, 5000)
Xte, Yte = features(big_test,  1000)
print(Xtr.shape, Xte.shape)

In [ ]:
def train_head(X, Y, epochs=30, Xv=None, Yv=None):
    """512칸으로 Linear(512, 10) 하나만 학습시켜 (모델, 정확도) 를 돌려준다"""
    Xv = Xv if Xv is not None else Xte
    Yv = Yv if Yv is not None else Yte
    torch.manual_seed(42)
    h = nn.Linear(512, 10)
    o = torch.optim.Adam(h.parameters(), lr=0.001)
    f = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 128):
            b = perm[i:i + 128]
            o.zero_grad()
            f(h(X[b]), Y[b]).backward()
            o.step()
    with torch.no_grad():
        return h, (h(Xv).argmax(1) == Yv).float().mean().item()

head, acc = train_head(Xtr, Ytr)
print('전이학습 정확도 %.4f  학습한 계수 %d' % (acc, count(head)))

> **실습문제 7.** 마지막 층을 **10 종류짜리**로 갈아 끼운다. 입력 칸 수 `512` 는 앞쪽 층이 내놓는 값이라 바꿀 수 없다.

In [ ]:
net.fc = nn.Linear(512, 10)
print(net.fc)
print('새 층의 계수', sum(p.numel() for p in net.fc.parameters()))

n = sum(p.numel() for p in net.fc.parameters())
assert n == 5130, f'512*10+10 = 5,130 이어야 한다: {n}'

> **실습문제 8.** 준비 셀의 `train_head` 안쪽 루프를 **직접 써서** `my_head` 를 학습시킨다. 1주차 학습 루프 다섯 줄과 같고, 사진 대신 `Xtr` 의 512칸을 먹인다는 점만 다르다.

In [ ]:
torch.manual_seed(42)
my_head = nn.Linear(512, 10)
opt = torch.optim.Adam(my_head.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for e in range(30):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        b = perm[i:i + 128]
        opt.zero_grad()                          # 기울기를 비운다
        loss_fn(my_head(Xtr[b]), Ytr[b]).backward()   # 기울기를 구한다
        opt.step()                          # 한 걸음 옮긴다

with torch.no_grad():
    my_acc = (my_head(Xte).argmax(1) == Yte).float().mean().item()
print('%.4f  학습한 계수 %d' % (my_acc, count(my_head)))

assert count(my_head) == 5130, f'512*10+10 = 5,130 이어야 한다: {count(my_head)}'
assert abs(my_acc - acc) < 1e-6, f'준비 셀과 같아야 한다: {my_acc} 대 {acc}'
print('계수 5,130개로 %.4f — CNN 60,362개보다 적게 배우고 더 맞혔다' % my_acc)

## 5. 세 방식을 한 표로

> **실습문제 9.** 오늘 만든 세 방식을 **한 표**로 정리해 찍는다. `방식 · 학습한 계수 · 정확도` 세 칸이다. 전이학습은 테스트 500장, 나머지는 1,000장이라 조건이 다른 것을 표 아래에 적는다.

In [ ]:
print('%-22s %10s %10s' % ('방식', '학습 계수', '정확도'))
print('-' * 44)
print('%-22s %10d %10.4f' % ('펴서 Linear', count(mlp), score(mlp)))
print('%-22s %10d %10.4f' % ('CNN 처음부터', count(cnn), score(cnn)))
print('%-22s %10d %10.4f' % ('resnet18 특징 + Linear', 5130, acc))
print()
print('훈련 5,000장 · 테스트 1,000장 — 세 방식 모두 같은 조건이다')
print('계수는 줄고 정확도는 오른다 — 이것이 오늘의 결론이다')